In [47]:
import os
import random
import shutil
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter
import noise

# Validation Flow

In [48]:
def generate_defect_mask(
    height: int, width: int, rng: random.Random, center: tuple = None
) -> np.ndarray:
    """
    Generate a single connected binary defect mask using Perlin noise.

    Multi-octave Perlin noise is evaluated on a grid centered at ``center``,
    then Gaussian-weighted so only one coherent blob region exceeds the
    threshold. The result is an organic, fractal-edged defect region that
    closely resembles real anomaly shapes (scratches, contamination patches).

    Args:
        height: Mask height in pixels.
        width:  Mask width in pixels.
        rng:    Seeded random instance for reproducibility.
        center: (cy, cx) pixel coordinates for the defect center.
                If None, a random point in the central 60% of the image is used.

    Returns:
        Float32 numpy array of shape (height, width) with values in {0.0, 1.0},
        where 1.0 marks defect pixels.
    """
    if center is None:
        cy = rng.uniform(0.2, 0.8) * height
        cx = rng.uniform(0.2, 0.8) * width
    else:
        cy, cx = float(center[0]), float(center[1])

    # Perlin noise parameters
    scale = rng.uniform(2.0, 6.0)       # lower → larger, smoother blobs
    octaves = rng.randint(4, 8)          # more octaves → more fractal detail
    persistence = rng.uniform(0.4, 0.7)  # amplitude falloff per octave
    lacunarity = rng.uniform(1.8, 2.2)   # frequency multiplier per octave
    base = rng.randint(0, 255)           # random seed offset for noise field

    perlin = np.zeros((height, width), dtype=np.float32)
    for y in range(height):
        for x in range(width):
            nx = x / width * scale
            ny = y / height * scale
            perlin[y, x] = noise.pnoise2(
                nx, ny,
                octaves=octaves,
                persistence=persistence,
                lacunarity=lacunarity,
                base=base,
            )

    # Normalize to [0, 1]
    perlin = (perlin - perlin.min()) / (perlin.max() - perlin.min() + 1e-8)

    # Gaussian weight centered at (cy, cx) → single connected blob
    sigma = rng.uniform(0.06, 0.09) * min(height, width)
    yy, xx = np.mgrid[0:height, 0:width]
    weight = np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * sigma ** 2))

    combined = perlin * weight

    # Threshold to target defect coverage of 0.01–2% of image area
    target_coverage = rng.uniform(0.0001, 0.02)
    threshold = float(np.percentile(combined, (1.0 - target_coverage) * 100))
    mask = (combined >= threshold).astype(np.float32)
    return mask

In [49]:
def inject_defect(
    image: Image.Image,
    texture_dirs: list,
    rng: random.Random,
) -> tuple:
    """
    Inject a synthetic defect into ``image`` using texture blending.

    The defect center is sampled from foreground pixels only (detected by
    comparing each pixel against the background color estimated from the image
    corners). The generated defect mask is then intersected with the foreground
    mask so no defect pixel can land on the background.

    A random texture patch is blended in within the mask:
        result = image * (1 - mask) + texture * mask

    Args:
        image:        PIL RGB image to corrupt.
        texture_dirs: List of directories from which to sample texture images.
                      Typically the ``train/good`` folders of other MVTec categories.
        rng:          Seeded random instance for reproducibility.

    Returns:
        (defected_image, mask_image) — PIL RGB image with the injected defect and
        a grayscale PIL Image (mode "L") of the binary defect mask (0 or 255).
    """
    w, h = image.size
    img_arr = np.array(image, dtype=np.float32) / 255.0

    # Estimate background color from corners and build a foreground mask
    cs = max(5, min(h, w) // 20)
    corner_pixels = np.concatenate([
        img_arr[:cs, :cs].reshape(-1, 3),
        img_arr[:cs, -cs:].reshape(-1, 3),
        img_arr[-cs:, :cs].reshape(-1, 3),
        img_arr[-cs:, -cs:].reshape(-1, 3),
    ], axis=0)
    bg_color = corner_pixels.mean(axis=0)
    diff = np.linalg.norm(img_arr - bg_color, axis=2)
    fg_mask = (diff > 0.1).astype(np.float32)

    # Pick the defect center from a foreground pixel
    fg_pixels = np.argwhere(fg_mask > 0)
    if len(fg_pixels) > 0:
        cy, cx = fg_pixels[rng.randint(0, len(fg_pixels) - 1)]
    else:
        cy, cx = h // 2, w // 2

    # Sample a texture image from a randomly chosen texture directory
    texture_dir = rng.choice(texture_dirs)
    texture_files = sorted(
        f for f in os.listdir(texture_dir)
        if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp"))
    )
    if not texture_files:
        raise ValueError(f"No texture images found in: {texture_dir}")

    texture_path = os.path.join(texture_dir, rng.choice(texture_files))
    texture_arr = np.array(
        Image.open(texture_path).convert("RGB").resize((w, h), Image.BILINEAR),
        dtype=np.float32,
    ) / 255.0

    # Generate blob mask then clip it to the foreground — no defect on background
    mask = generate_defect_mask(h, w, rng, center=(cy, cx))
    mask = mask * fg_mask

    mask_3d = mask[:, :, np.newaxis]
    defected_arr = img_arr * (1.0 - mask_3d) + texture_arr * mask_3d
    defected_arr = np.clip(defected_arr * 255, 0, 255).astype(np.uint8)
    mask_uint8 = (mask * 255).astype(np.uint8)

    return Image.fromarray(defected_arr), Image.fromarray(mask_uint8)

In [50]:
def augment(
    image: Image.Image,
    rng: random.Random,
    mask: Image.Image = None,
    target_size: tuple = (256, 256),
) -> tuple:
    """
    Apply a reproducible augmentation pipeline to ``image`` (and optionally ``mask``).

    Serves two purposes:
      1. Anti-lookup: geometric transforms (flip, rotation, crop) ensure the
         final image differs from any publicly known ground-truth pixel data,
         making it infeasible for miners to match images to known MVTec samples.
      2. Edge simulation: photometric transforms (noise, brightness/contrast, blur)
         mimic the imaging conditions on Raspberry Pi / Jetson edge cameras.

    Geometric transforms are applied identically to both image and mask to
    preserve pixel-level correspondence. Photometric transforms are image-only.
    The final step resizes everything to ``target_size`` (canonical 256×256).

    Args:
        image:       PIL RGB image.
        rng:         Seeded random instance — same seed as the rest of the
                     pipeline so transforms are fully reproducible per round.
        mask:        Optional grayscale PIL mask (mode "L"). Geometric transforms
                     are mirrored exactly using nearest-neighbour resampling so
                     binary mask values are never interpolated.
        target_size: (width, height) of the output. Default 256×256.

    Returns:
        (aug_image, aug_mask) — aug_mask is None when no mask was provided.
    """
    # ---------------------------------------------------------------- geometric
    # 1. Random horizontal flip
    if rng.random() < 0.5:
        image = image.transpose(Image.FLIP_LEFT_RIGHT)
        if mask is not None:
            mask = mask.transpose(Image.FLIP_LEFT_RIGHT)

    # 2. Random vertical flip
    if rng.random() < 0.5:
        image = image.transpose(Image.FLIP_TOP_BOTTOM)
        if mask is not None:
            mask = mask.transpose(Image.FLIP_TOP_BOTTOM)

    # 3. Random rotation ±15° — bilinear for image, nearest for mask
    angle = rng.uniform(-15, 15)
    image = image.rotate(angle, resample=Image.BILINEAR, expand=False)
    if mask is not None:
        mask = mask.rotate(angle, resample=Image.NEAREST, expand=False)

    # 4. Random crop (80–100% of image) then resize back to original dimensions
    w, h = image.size
    crop_scale = rng.uniform(0.8, 1.0)
    crop_w, crop_h = int(w * crop_scale), int(h * crop_scale)
    x0 = rng.randint(0, w - crop_w)
    y0 = rng.randint(0, h - crop_h)
    box = (x0, y0, x0 + crop_w, y0 + crop_h)
    image = image.crop(box).resize((w, h), Image.BILINEAR)
    if mask is not None:
        mask = mask.crop(box).resize((w, h), Image.NEAREST)

    # 5. Resize to canonical target resolution
    image = image.resize(target_size, Image.BILINEAR)
    if mask is not None:
        mask = mask.resize(target_size, Image.NEAREST)

    # ----------------------------------------------------------- photometric
    # 6. Gaussian sensor noise  σ ∈ [0, 8] / 255
    noise_sigma = rng.uniform(0.0, 8.0) / 255.0
    if noise_sigma > 0:
        np_rng = np.random.RandomState(rng.randint(0, 2**31 - 1))
        img_arr = np.array(image, dtype=np.float32) / 255.0
        img_arr += np_rng.randn(*img_arr.shape).astype(np.float32) * noise_sigma
        image = Image.fromarray(np.clip(img_arr * 255, 0, 255).astype(np.uint8))

    # 7. Brightness jitter ±15%
    image = ImageEnhance.Brightness(image).enhance(rng.uniform(0.85, 1.15))

    # 8. Contrast jitter ±15%
    image = ImageEnhance.Contrast(image).enhance(rng.uniform(0.85, 1.15))

    # 9. Optional mild Gaussian blur (50% chance, radius 0–1.5 px)
    if rng.random() < 0.5:
        image = image.filter(ImageFilter.GaussianBlur(radius=rng.uniform(0.0, 1.5)))

    return image, mask

In [ ]:
def generate_validation_dataset(
    src_dataset_path: str,
    output_path: str,
    dtd_path: str,
    num_images: int = 20,
    seed: int = 42,
    p: float = 0.5,
    real_defect_ratio: float = 0.3,
) -> None:
    """
    Generate a hybrid validation dataset from the MVTec_AD bottle category.

    In total ``num_images`` images are produced. With overall probability ``p``
    an image is defective; of those defective images, ``real_defect_ratio``
    come from the real MVTec test split (with their original ground-truth masks)
    and the rest are synthetically generated (Perlin-noise mask + DTD texture
    blend).  Good images are sampled from the train/good split.
    Augmentation is applied to all images as the last step before saving.

    Output layout:
        output_path/
            ground_truth/
                synthetic/              # masks for synthetic defects
                broken_large/           # original MVTec masks (renamed *_mask.png)
                broken_small/
                contamination/
            test/
                good/                   # good images
                synthetic/              # synthetic defective images
                broken_large/           # real defective images
                broken_small/
                contamination/

    Args:
        src_dataset_path:  Path to the MVTec_AD root (contains ``bottle/``).
        output_path:       Destination folder for the generated dataset.
        dtd_path:          Path to the DTD dataset root (contains ``images/``).
        num_images:        Total number of images in the output dataset.
        seed:              Random seed — should be derived from the block hash
                           each validation round so miners cannot memorise the set.
        p:                 Overall probability that any given image is defective.
        real_defect_ratio: Fraction of defective images sourced from the real
                           MVTec test split (0.0 = all synthetic, 1.0 = all real).
    """
    # ------------------------------------------------------------------ paths
    good_train_dir = os.path.join(src_dataset_path, "bottle", "train", "good")
    test_dir = os.path.join(src_dataset_path, "bottle", "test")
    gt_dir = os.path.join(src_dataset_path, "bottle", "ground_truth")
    real_defect_categories = ["broken_large", "broken_small", "contamination"]

    for d in [good_train_dir, test_dir, gt_dir]:
        if not os.path.isdir(d):
            raise FileNotFoundError(f"Required directory not found: {d}")

    dtd_images_dir = os.path.join(dtd_path, "images")
    if not os.path.isdir(dtd_images_dir):
        raise FileNotFoundError(f"DTD images directory not found: {dtd_images_dir}")

    texture_dirs = sorted(
        os.path.join(dtd_images_dir, cat)
        for cat in os.listdir(dtd_images_dir)
        if os.path.isdir(os.path.join(dtd_images_dir, cat))
    )
    if not texture_dirs:
        raise RuntimeError(f"No texture categories found in: {dtd_images_dir}")

    # ----------------------------------------------------------------- wipe output
    # Always start from a clean slate so no stale files from previous runs survive.
    if os.path.exists(output_path):
        shutil.rmtree(output_path)

    # ------------------------------------------- compute split sizes
    rng = random.Random(seed)
    num_defective = round(num_images * p)
    num_good = num_images - num_defective
    num_real = round(num_defective * real_defect_ratio)
    num_synthetic = num_defective - num_real

    # ------------------------------------------- create output dirs
    def make_dirs(*parts):
        path = os.path.join(output_path, *parts)
        os.makedirs(path, exist_ok=True)
        return path

    test_good_dir = make_dirs("test", "good")
    test_syn_dir = make_dirs("test", "synthetic")
    gt_syn_dir = make_dirs("ground_truth", "synthetic")
    real_out_dirs = {
        cat: (make_dirs("test", cat), make_dirs("ground_truth", cat))
        for cat in real_defect_categories
    }

    # ------------------------------------------- good images (train/good)
    train_good_files = sorted(
        f for f in os.listdir(good_train_dir)
        if os.path.isfile(os.path.join(good_train_dir, f))
    )
    selected_good = rng.sample(train_good_files, min(num_good, len(train_good_files)))
    for filename in selected_good:
        src = os.path.join(good_train_dir, filename)
        image = Image.open(src).convert("RGB")
        image, _ = augment(image, rng)
        image.save(os.path.join(test_good_dir, filename))

    # ------------------------------------------- synthetic defective images
    synthetic_pool = [f for f in train_good_files if f not in selected_good]
    selected_synthetic = rng.sample(synthetic_pool, min(num_synthetic, len(synthetic_pool)))
    for filename in selected_synthetic:
        src = os.path.join(good_train_dir, filename)
        image = Image.open(src).convert("RGB")
        defected_image, mask_image = inject_defect(image, texture_dirs, rng)
        defected_image, mask_image = augment(defected_image, rng, mask=mask_image)
        defected_image.save(os.path.join(test_syn_dir, filename))
        mask_image.save(os.path.join(gt_syn_dir, filename))

    # ------------------------------------------- real defective images
    # Gather all real defect samples across categories
    real_pool = []
    for cat in real_defect_categories:
        cat_test_dir = os.path.join(test_dir, cat)
        for f in sorted(os.listdir(cat_test_dir)):
            if os.path.isfile(os.path.join(cat_test_dir, f)):
                real_pool.append((cat, f))

    selected_real = rng.sample(real_pool, min(num_real, len(real_pool)))
    for cat, filename in selected_real:
        stem = os.path.splitext(filename)[0]
        mask_filename = f"{stem}_mask.png"

        src_img = os.path.join(test_dir, cat, filename)
        src_mask = os.path.join(gt_dir, cat, mask_filename)

        image = Image.open(src_img).convert("RGB")
        mask_pil = Image.open(src_mask).convert("L")
        image, mask_pil = augment(image, rng, mask=mask_pil)
        test_cat_dir, gt_cat_dir = real_out_dirs[cat]
        image.save(os.path.join(test_cat_dir, filename))
        mask_pil.save(os.path.join(gt_cat_dir, mask_filename))

    print(
        f"Validation dataset created at '{output_path}':\n"
        f"  {num_good} good images (train split)\n"
        f"  {len(selected_synthetic)} synthetic defective images\n"
        f"  {len(selected_real)} real defective images "
        f"({', '.join(f'{sum(1 for c,_ in selected_real if c==cat)} {cat}' for cat in real_defect_categories)})\n"
        f"  seed={seed}"
    )

In [52]:
generate_validation_dataset(
    src_dataset_path="../data/datasets/MVTec_AD",
    output_path="../data/datasets/validation_dataset",
    dtd_path="../data/datasets/dtd",
    num_images=50,
    seed=42,
    p=0.5,
    real_defect_ratio=0.3,
)

Validation dataset created at '../data/datasets/validation_dataset':
  25 good images (train split)
  17 synthetic defective images
  8 real defective images (3 broken_large, 2 broken_small, 3 contamination)
  seed=42


# Incentive Mechanism

## Overview

Miners submit trained **model files** (weights + architecture descriptor). The validator downloads each model, loads it into a sandboxed runtime, and runs inference locally on the validation dataset. This means the speed and deployability measurements are ground truth — the validator actually executes the model on reference hardware rather than trusting a self-reported latency.

The validator generates a fresh hybrid dataset each round (seed derived from block hash) and runs every miner's model against the same images. Models are scored on three axes:

| Component | Weight | Metric |
|---|---|---|
| Detection Accuracy | 50% | Per-image reward (classification + optional segmentation) |
| Inference Speed | 30% | Actual ms/image measured by the validator on reference hardware |
| Robustness | 20% | Prediction consistency under augmented probe images |

---


## Per-image Loss

$$L_i = \alpha_{cls} \cdot L^{cls}_i \;+\; \alpha_{loc} \cdot L^{loc}_i$$

### Classification loss $L^{cls}_i$

The validator reads the raw sigmoid output $\hat{p}_i \in [0,1]$ directly from the model. Normalised binary cross-entropy keeps the loss in $[0, 1]$:

$$L^{cls}_i = \frac{-\bigl[y_i \log \hat{p}_i + (1-y_i)\log(1-\hat{p}_i)\bigr]}{\log 2}$$

### Localisation loss $L^{loc}_i$

Pixel IoU between the predicted mask $\hat{M}_i$ (thresholded at 0.5) and the ground-truth mask $M_i$:

$$L^{loc}_i = 1 - \frac{|\hat{M}_i \cap M_i|}{|\hat{M}_i \cup M_i|}$$

For **good images** ($M_i = \mathbf{0}$) the model is rewarded for outputting an all-zero mask.  
If **no `mask` head is present**, $L^{loc}_i = 1$ for every image (no partial credit).

Recommended component weights within the accuracy term:

| $\alpha_{cls}$ | $\alpha_{loc}$ |
|---|---|
| 0.6 | 0.4 |

---

## Per-image Reward

$$R_i = \alpha_{cls}(1 - L^{cls}_i) + \alpha_{loc}(1 - L^{loc}_i)$$

---

## Batch-level Accuracy Reward

$$\bar{R}_{acc} = \frac{1}{N} \sum_{i=1}^{N} R_i$$

F1 (macro) is also computed for the leaderboard display but $\bar{R}_{acc}$ drives on-chain weights.

---

## Speed Reward

The validator measures wall-clock time for a full forward pass on each image individually and takes the mean. Reference hardware is a **Raspberry Pi 4** (or equivalent 4-core ARM Cortex-A72, 4 GB RAM).

$$R_{speed} = \max\!\left(0,\; 1 - \frac{\bar{t} - t_{soft}}{t_{hard} - t_{soft}}\right)$$

| Parameter | Value |
|---|---|
| $t_{soft}$ | 200 ms / image |
| $t_{hard}$ | 1 000 ms / image |
| Above $t_{hard}$ | $R_{speed} = 0$ (but accuracy still scored) |

Because the validator runs the model itself, miners cannot self-report latency or use server-side acceleration to fake edge performance.

---

## Robustness Reward

The validator embeds $k$ augmented copies of $m$ **probe images** in the batch (the same augmentation pipeline used during dataset generation). A model that truly learned the signal predicts consistently across all $k$ variants; a model that overfits to exact pixel values will be inconsistent.

$$R_{robust} = \frac{1}{m} \sum_{j=1}^{m} \mathbf{1}\!\left[\text{all } k \text{ predictions for probe } j \text{ agree}\right]$$

For models with a mask head, "agree" is relaxed to mean pairwise IoU $> 0.8$ across the $k$ predictions.

---

## Total Reward

$$\boxed{R_{total} = \underbrace{0.5}_{\alpha_{acc}} \cdot \bar{R}_{acc} \;+\; \underbrace{0.3}_{\alpha_{speed}} \cdot R_{speed} \;+\; \underbrace{0.2}_{\alpha_{robust}} \cdot R_{robust}}$$

---

## Anti-gaming Properties

| Threat | Mitigation |
|---|---|
| Look up MVTec ground truth online | Geometric augmentation produces unique pixels every round — a lookup table cannot match the transformed images |
| Memorise prior-round datasets | Seed rotates every round from block hash — different image selection and transforms each time |
| Submit a large accurate model with no edge concern | Deployability gate hard-fails models exceeding RAM/size/latency limits |
| Use cloud GPU for inference, report low latency | Validator runs the model itself on reference hardware — self-reported latency is never trusted |
| Skip mask head, win on classification alone | No mask → $L^{loc}_i = 1$ always → max achievable score is $0.6 \times 0.5 = 0.30$ of total |

---

## Reward Implementation

In [53]:
import math
from typing import Dict, List, Optional
import numpy as np

# --------------------------------------------------------------------------- #
#  Component weights (mirror the spec)
# --------------------------------------------------------------------------- #
ALPHA_ACC     = 0.5   # accuracy share of total reward
ALPHA_SPEED   = 0.3   # speed share
ALPHA_ROBUST  = 0.2   # robustness share

ALPHA_CLS     = 0.6   # classification share within accuracy
ALPHA_LOC     = 0.4   # localisation share within accuracy

T_SOFT        = 0.200  # seconds/image — no speed penalty below this
T_HARD        = 1.000  # seconds/image — zero speed reward at or above this

MASK_AGREE_IOU = 0.8   # minimum IoU to call two mask predictions "in agreement"


# --------------------------------------------------------------------------- #
#  1. Classification reward
# --------------------------------------------------------------------------- #
def classification_reward(label: int, confidence: Optional[float]) -> float:
    """
    Reward for the binary defect/good classification head.

    If the model outputs a raw sigmoid confidence score, normalised binary
    cross-entropy is used so the reward stays in [0, 1]:

        reward = 1 - BCE(label, confidence) / log(2)

    Without a confidence score only a hard match (0.0 or 1.0) is returned.

    Args:
        label:      Ground-truth label (1 = defective, 0 = good).
        confidence: Raw sigmoid output from the model, or None if unavailable.

    Returns:
        float in [0, 1].
    """
    if confidence is None:
        return 0.0  # maximum loss — caller should supply at least a hard label

    # Clip to avoid log(0)
    p = float(np.clip(confidence, 1e-7, 1 - 1e-7))
    y = float(label)
    bce = -(y * math.log(p) + (1 - y) * math.log(1 - p))
    # Normalise: maximum BCE (for a completely wrong prediction) is -log(ε) ≈ log(2)
    # Dividing by log(2) maps [0, log(2)] → [0, 1].
    return float(np.clip(1.0 - bce / math.log(2), 0.0, 1.0))


# --------------------------------------------------------------------------- #
#  2. Mask IoU reward
# --------------------------------------------------------------------------- #
def mask_iou_reward(gt_mask: Optional[np.ndarray], pred_mask: Optional[np.ndarray]) -> float:
    """
    Pixel-level Intersection-over-Union reward for the segmentation head.

    Both masks are expected to be binary (0/1 or bool) numpy arrays of the same
    shape (256 × 256).  A model that has no mask head receives 0.0 (maximum
    loss) — there is no partial credit for classification alone.

    Special case: if the ground-truth mask is all-zero (good image) the model
    is rewarded for returning an all-zero prediction:
        - all-zero prediction → reward = 1.0
        - any positive pixel  → reward = 1 - FP_rate

    Args:
        gt_mask:   Ground-truth binary mask or None.
        pred_mask: Predicted binary mask or None.

    Returns:
        float in [0, 1].  0.0 when pred_mask is None.
    """
    if pred_mask is None:
        return 0.0

    gt   = (np.asarray(gt_mask,   dtype=bool) if gt_mask   is not None else np.zeros((256, 256), dtype=bool))
    pred = (np.asarray(pred_mask, dtype=bool))

    intersection = np.logical_and(gt, pred).sum()
    union        = np.logical_or(gt, pred).sum()

    if union == 0:
        # Both masks are empty — perfect agreement on a good image.
        return 1.0

    return float(intersection) / float(union)


# --------------------------------------------------------------------------- #
#  3. Per-image reward
# --------------------------------------------------------------------------- #
def image_reward(
    label: int,
    confidence: Optional[float],
    gt_mask: Optional[np.ndarray],
    pred_mask: Optional[np.ndarray],
    alpha_cls: float = ALPHA_CLS,
    alpha_loc: float = ALPHA_LOC,
    verbose: bool = False,
) -> Dict[str, float]:
    """
    Combined per-image reward.

        R_i = α_cls * (1 - L_cls) + α_loc * (1 - L_loc)

    Args:
        label:      Ground-truth label (1 = defective, 0 = good).
        confidence: Raw sigmoid output from the model (None if not returned).
        gt_mask:    Ground-truth binary 256×256 mask (None for good images is fine).
        pred_mask:  Predicted binary 256×256 mask (None if model has no mask head).
        alpha_cls:  Weight for the classification component.
        alpha_loc:  Weight for the localisation component.
        verbose:    Print component scores.

    Returns:
        dict with keys 'cls', 'loc', 'total'.
    """
    cls_r = classification_reward(label, confidence)
    loc_r = mask_iou_reward(gt_mask, pred_mask)

    total = (alpha_cls * cls_r + alpha_loc * loc_r) / (alpha_cls + alpha_loc)

    result = {"cls": cls_r, "loc": loc_r, "total": total}
    if verbose:
        print(", ".join(f"{k}: {v:.3f}" for k, v in result.items()))
    return result


# --------------------------------------------------------------------------- #
#  4. Speed reward
# --------------------------------------------------------------------------- #
def speed_reward(
    mean_latency_s: float,
    t_soft: float = T_SOFT,
    t_hard: float = T_HARD,
) -> float:
    """
    Linear speed reward that decays from 1.0 to 0.0 between the soft and hard
    latency thresholds (seconds per image, measured by the validator).

        R_speed = max(0, 1 - (t - t_soft) / (t_hard - t_soft))

    Args:
        mean_latency_s: Mean per-image inference time in seconds.
        t_soft:         Below this latency the full speed reward is given.
        t_hard:         At or above this latency the speed reward is zero.

    Returns:
        float in [0, 1].
    """
    if mean_latency_s <= t_soft:
        return 1.0
    if mean_latency_s >= t_hard:
        return 0.0
    return 1.0 - (mean_latency_s - t_soft) / (t_hard - t_soft)


# --------------------------------------------------------------------------- #
#  5. Robustness reward
# --------------------------------------------------------------------------- #
def robustness_reward(
    probe_predictions: List[List[Optional[float]]],
    probe_masks: Optional[List[List[Optional[np.ndarray]]]] = None,
    mask_agree_iou: float = MASK_AGREE_IOU,
) -> float:
    """
    Reward consistency across k augmented copies of each probe image.

    The validator embeds k augmented variants of m probe images in the batch.
    A miner model that has truly learned the signal will predict the same label
    regardless of the specific augmentation applied.

        R_robust = (1/m) * Σ_j  𝟙[all k predictions for probe j agree]

    For models with a mask head, "agree" means pairwise IoU > mask_agree_iou.

    Args:
        probe_predictions: List of m groups, each group being k sigmoid
                           confidence values (or None) for that probe.
                           Shape: [m][k].
        probe_masks:       Optional list of m groups, each group being k binary
                           256×256 masks (or None).  Shape: [m][k].
        mask_agree_iou:    IoU threshold above which two masks are considered
                           in agreement.

    Returns:
        float in [0, 1].
    """
    m = len(probe_predictions)
    if m == 0:
        return 1.0  # no probes → don't penalise

    agreed = 0
    for j, group in enumerate(probe_predictions):
        # Hard label agreement (threshold at 0.5)
        hard_labels = [int((c or 0.0) >= 0.5) for c in group]
        label_agree = len(set(hard_labels)) == 1

        # Mask agreement (if provided)
        mask_agree = True
        if probe_masks is not None and j < len(probe_masks):
            masks = [m for m in probe_masks[j] if m is not None]
            if len(masks) >= 2:
                pairwise_ious = [
                    mask_iou_reward(masks[a], masks[b])
                    for a in range(len(masks))
                    for b in range(a + 1, len(masks))
                ]
                mask_agree = all(iou >= mask_agree_iou for iou in pairwise_ious)

        if label_agree and mask_agree:
            agreed += 1

    return agreed / m


# --------------------------------------------------------------------------- #
#  6. Top-level reward  (called once per miner per round)
# --------------------------------------------------------------------------- #
def reward(
    ground_truth: List[Dict],
    predictions: List[Dict],
    mean_latency_s: float,
    probe_predictions: Optional[List[List[Optional[float]]]] = None,
    probe_masks: Optional[List[List[Optional[np.ndarray]]]] = None,
    alpha_acc: float = ALPHA_ACC,
    alpha_speed: float = ALPHA_SPEED,
    alpha_robust: float = ALPHA_ROBUST,
    verbose: bool = False,
) -> float:
    """
    Score a miner's model for one validation round.

    Args:
        ground_truth:      List of N dicts, one per image:
                             {"image_id": str, "label": int,
                              "mask": np.ndarray | None}
        predictions:       List of N dicts returned by the miner's model:
                             {"image_id": str,
                              "confidence": float | None,
                              "mask": np.ndarray | None}
                           Order must match ground_truth (matched by image_id).
        mean_latency_s:    Mean per-image inference time (seconds) measured by
                           the validator on reference hardware.
        probe_predictions: Robustness probe confidences — see robustness_reward().
                           Pass None to skip the robustness component.
        probe_masks:       Robustness probe masks — see robustness_reward().
        alpha_acc:         Weight for the accuracy component.
        alpha_speed:       Weight for the speed component.
        alpha_robust:      Weight for the robustness component.
        verbose:           Print per-image and aggregate scores.

    Returns:
        float in [0, 1].  0.0 if predictions is None or empty.
    """
    if not predictions:
        return 0.0

    # Index predictions by image_id for order-independent matching
    pred_index = {p["image_id"]: p for p in predictions}

    per_image = []
    for gt in ground_truth:
        pred = pred_index.get(gt["image_id"], {})
        r = image_reward(
            label=gt["label"],
            confidence=pred.get("confidence"),
            gt_mask=gt.get("mask"),
            pred_mask=pred.get("mask"),
            verbose=verbose,
        )
        per_image.append(r["total"])

    acc_r    = float(np.mean(per_image))
    speed_r  = speed_reward(mean_latency_s)
    robust_r = (
        robustness_reward(probe_predictions, probe_masks)
        if probe_predictions is not None
        else 1.0  # no probes provided → full robustness credit
    )

    total_w  = alpha_acc + alpha_speed + alpha_robust
    total_r  = (alpha_acc * acc_r + alpha_speed * speed_r + alpha_robust * robust_r) / total_w

    if verbose:
        print(
            f"acc_reward: {acc_r:.3f}, "
            f"speed_reward: {speed_r:.3f}, "
            f"robust_reward: {robust_r:.3f}, "
            f"total_reward: {total_r:.3f}"
        )

    return float(total_r)

In [54]:
# ── Smoke test ──────────────────────────────────────────────────────────────
rng_t = random.Random(0)

# Build a tiny synthetic ground-truth batch (5 images)
gt_batch = []
pred_batch = []
for i in range(5):
    lbl  = rng_t.randint(0, 1)
    mask = None
    if lbl:
        m = np.zeros((256, 256), dtype=np.uint8)
        m[80:140, 100:160] = 1   # small defect region
        mask = m

    gt_batch.append({"image_id": f"img_{i:03d}.png", "label": lbl, "mask": mask})

    # Simulate a reasonably good model: correct label most of the time,
    # slightly noisy confidence, mask slightly shifted.
    conf  = (0.85 + rng_t.uniform(-0.1, 0.1)) if lbl else (0.10 + rng_t.uniform(-0.05, 0.05))
    pred_mask = None
    if lbl and mask is not None:
        pm = np.zeros((256, 256), dtype=np.uint8)
        pm[85:145, 105:165] = 1  # shifted by ~5 px
        pred_mask = pm

    pred_batch.append({"image_id": f"img_{i:03d}.png", "confidence": conf, "mask": pred_mask})

# Robustness probes: 2 probe images × 3 augmented variants each
probe_preds = [
    [0.88, 0.91, 0.86],   # probe 0 — consistent defective
    [0.12, 0.09, 0.55],   # probe 1 — inconsistent (would fail robustness)
]

total = reward(
    ground_truth=gt_batch,
    predictions=pred_batch,
    mean_latency_s=0.150,       # 150 ms/image — below soft threshold
    probe_predictions=probe_preds,
    verbose=True,
)
print(f"\nFinal miner reward: {total:.4f}")

cls: 0.851, loc: 0.725, total: 0.800
cls: 0.600, loc: 0.725, total: 0.650
cls: 0.733, loc: 0.725, total: 0.730
cls: 0.916, loc: 0.725, total: 0.840
cls: 0.794, loc: 0.725, total: 0.766
acc_reward: 0.757, speed_reward: 1.000, robust_reward: 0.500, total_reward: 0.779

Final miner reward: 0.7785


# Miner Comparison Demo

Load both trained ONNX models, run them against the validation dataset, and compare rewards side-by-side.

| Model | Architecture | Mask head? | Expected advantage |
|---|---|---|---|
| Baseline | MobileNetV2 classifier | No | Speed only |
| Improved | MobileNetV2 + U-Net decoder | Yes | Accuracy + localisation |

In [55]:
import time
import onnxruntime as ort

# ── Paths ──────────────────────────────────────────────────────────────────
BASELINE_ONNX  = "../training/models/baseline.onnx"
IMPROVED_ONNX  = "../training/models/improved.onnx"
VAL_DATASET    = "../data/datasets/validation_dataset"
MVTEC_ROOT     = "../data/datasets/MVTec_AD"

# ImageNet normalisation constants (must match training)
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
TARGET = (256, 256)


def _load_image(path: str) -> np.ndarray:
    """Load a 256×256 RGB image as a normalised [1, 3, 256, 256] float32 array."""
    img = Image.open(path).convert("RGB").resize(TARGET, Image.BILINEAR)
    arr = np.array(img, dtype=np.float32) / 255.0          # [H, W, 3]
    arr = (arr - MEAN) / STD
    return arr.transpose(2, 0, 1)[np.newaxis]               # [1, 3, H, W]


def _load_mask(path: str) -> np.ndarray:
    """Load a ground-truth mask as a binary [256, 256] uint8 array (0/1)."""
    return (np.array(Image.open(path).convert("L").resize(TARGET, Image.NEAREST)) > 127).astype(np.uint8)


def _build_ground_truth(val_root: str) -> list:
    """
    Walk the validation_dataset directory and build a ground-truth list:
        [{"image_id": str, "label": int, "mask": np.ndarray | None}, ...]
    """
    gt = []
    test_root = os.path.join(val_root, "test")
    gt_root   = os.path.join(val_root, "ground_truth")

    for category in sorted(os.listdir(test_root)):
        cat_dir = os.path.join(test_root, category)
        if not os.path.isdir(cat_dir):
            continue
        is_good = (category == "good")
        for fname in sorted(os.listdir(cat_dir)):
            if not fname.lower().endswith((".png", ".jpg", ".jpeg")):
                continue
            image_id = f"{category}/{fname}"
            mask     = None
            if not is_good:
                stem      = os.path.splitext(fname)[0]
                mask_path = os.path.join(gt_root, category, f"{stem}_mask.png")
                if os.path.exists(mask_path):
                    mask = _load_mask(mask_path)
                else:
                    # synthetic defects: mask stored with the same filename
                    mask_path2 = os.path.join(gt_root, category, fname)
                    if os.path.exists(mask_path2):
                        mask = _load_mask(mask_path2)
            gt.append({"image_id": image_id, "label": 0 if is_good else 1, "mask": mask})

    return gt


def run_model(session: ort.InferenceSession, val_root: str, ground_truth: list) -> tuple:
    """
    Run all images through ``session`` and return (predictions, mean_latency_s).

    predictions: [{"image_id": str, "confidence": float, "mask": np.ndarray | None}]
    """
    output_names  = [o.name for o in session.get_outputs()]
    has_mask_head = "mask" in output_names

    predictions = []
    latencies   = []

    test_root = os.path.join(val_root, "test")
    for entry in ground_truth:
        category, fname = entry["image_id"].split("/", 1)
        img_path = os.path.join(test_root, category, fname)
        img      = _load_image(img_path)

        t0      = time.perf_counter()
        outputs = session.run(output_names, {"image": img})
        latencies.append(time.perf_counter() - t0)

        confidence = float(outputs[0].squeeze())
        pred_mask  = None
        if has_mask_head:
            raw = outputs[output_names.index("mask")].squeeze()  # [256, 256]
            pred_mask = (raw > 0.5).astype(np.uint8)

        predictions.append({
            "image_id":  entry["image_id"],
            "confidence": confidence,
            "mask":       pred_mask,
        })

    return predictions, float(np.mean(latencies))


print("Building ground truth …")
ground_truth = _build_ground_truth(VAL_DATASET)
print(f"  {len(ground_truth)} images  "
      f"({sum(1 for g in ground_truth if g['label']==0)} good, "
      f"{sum(1 for g in ground_truth if g['label']==1)} defective)")

Building ground truth …
  50 images  (25 good, 25 defective)


In [56]:
providers = ["CPUExecutionProvider"]

# ── Run baseline ──────────────────────────────────────────────────────────
print("Running baseline model …")
sess_b  = ort.InferenceSession(BASELINE_ONNX, providers=providers)
preds_b, lat_b = run_model(sess_b, VAL_DATASET, ground_truth)
score_b = reward(ground_truth, preds_b, lat_b, verbose=False)

# Decompose for the table
from functools import partial

def _acc_score(gt, preds):
    per = []
    idx = {p["image_id"]: p for p in preds}
    for g in gt:
        p = idx.get(g["image_id"], {})
        per.append(image_reward(g["label"], p.get("confidence"), g.get("mask"), p.get("mask"))["total"])
    return float(np.mean(per))

acc_b    = _acc_score(ground_truth, preds_b)
speed_b  = speed_reward(lat_b)
# no robustness probes in this demo → full credit awarded to both
robust_b = 1.0

# ── Run improved ──────────────────────────────────────────────────────────
print("Running improved model …")
sess_i  = ort.InferenceSession(IMPROVED_ONNX, providers=providers)
preds_i, lat_i = run_model(sess_i, VAL_DATASET, ground_truth)
score_i = reward(ground_truth, preds_i, lat_i, verbose=False)

acc_i    = _acc_score(ground_truth, preds_i)
speed_i  = speed_reward(lat_i)
robust_i = 1.0

# ── Comparison table ──────────────────────────────────────────────────────
from PIL import ImageDraw
import textwrap

header = f"{'Metric':<28} {'Baseline':>12} {'Improved':>12} {'Δ':>10}"
sep    = "-" * len(header)

def fmt(v): return f"{v:.4f}"
def delta(a, b): return f"{b-a:+.4f}"

rows = [
    ("Accuracy reward  (×0.50)",   acc_b,    acc_i),
    ("  Classification component", 
     np.mean([image_reward(g["label"], {p["image_id"]:p for p in preds_b}.get(g["image_id"],{}).get("confidence"), None, None)["cls"] for g in ground_truth]),
     np.mean([image_reward(g["label"], {p["image_id"]:p for p in preds_i}.get(g["image_id"],{}).get("confidence"), None, None)["cls"] for g in ground_truth]),
    ),
    ("  Localisation component",
     np.mean([mask_iou_reward(g.get("mask"), {p["image_id"]:p for p in preds_b}.get(g["image_id"],{}).get("mask")) for g in ground_truth]),
     np.mean([mask_iou_reward(g.get("mask"), {p["image_id"]:p for p in preds_i}.get(g["image_id"],{}).get("mask")) for g in ground_truth]),
    ),
    ("Speed reward     (×0.30)",   speed_b,  speed_i),
    (f"  Mean latency (s/image)",  lat_b,    lat_i),
    ("Robustness reward(×0.20)",   robust_b, robust_i),
    (sep, None, None),
    ("TOTAL REWARD",               score_b,  score_i),
]

print(f"\n{header}")
print(sep)
for name, vb, vi in rows:
    if vb is None:
        print(sep)
    else:
        print(f"{name:<28} {fmt(vb):>12} {fmt(vi):>12} {delta(vb, vi):>10}")

Running baseline model …
Running improved model …

Metric                           Baseline     Improved          Δ
-----------------------------------------------------------------
Accuracy reward  (×0.50)           0.5496       0.5893    +0.0397
  Classification component         0.9160       0.8973    -0.0187
  Localisation component           0.0000       0.1274    +0.1274
Speed reward     (×0.30)           1.0000       1.0000    +0.0000
  Mean latency (s/image)           0.0085       0.0321    +0.0236
Robustness reward(×0.20)           1.0000       1.0000    +0.0000
-----------------------------------------------------------------
TOTAL REWARD                       0.7748       0.7947    +0.0199
